# 00 — Exploración de archivos BIOM del EMP

Este notebook es un espacio de exploración para familiarizarse con los datos
del Earth Microbiome Project (EMP) antes de llevar la lógica estable a scripts
reutilizables.

**Objetivos concretos:**

- Entender qué contiene un archivo BIOM y cómo cargarlo con `biom-format`.
- Distinguir los dos ejes del BIOM: *muestras* (`sample`) y *observaciones* (`observation`).
- Entender qué son los ASVs y cómo se relacionan con la taxonomía.
- Ver la relación entre el archivo BIOM y el mapping file TSV.
- Extraer estadísticas por muestra (lecturas totales, ASVs observados).
- Construir un prototipo de `sample_table.csv` cruzando BIOM y mapping file.

**Lo que este notebook NO hace:**

- No limpia ni normaliza metadatos (eso será Fase 1).
- No construye el Knowledge Graph (eso será Fase 2).
- No convierte la matriz completa a formato denso (sería inviable: 155.002 ASVs × 2.000 muestras).

**Relación con el proyecto:**

Este notebook corresponde a la **Fase 0** del plan de trabajo: entender la
estructura de los datos antes de diseñar el pipeline. Toda la lógica que se
consolide aquí acabará en `build_sample_table.py` o en módulos de `src/empkg/`.

## 2. Imports y config inicial

Activa antes el entorno del proyecto:

```bash
conda activate empkg
jupyter lab
```

Cargamos las librerías que usaremos a lo largo del notebook.

- `pathlib.Path`: rutas multiplataforma, sin concatenar strings manualmente.
- `pandas`: para tablas de metadatos.
- `numpy`: para cálculos numéricos básicos.
- `biom`: librería oficial para leer archivos `.biom` (formato BIOM 2.x sobre HDF5).
- `h5py`: para inspeccionar la estructura HDF5 del BIOM directamente.
- `requests` y `tqdm`: para descargas con barra de progreso.

Si algún módulo no se importa, revisa que el entorno lo tenga definido.

In [294]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

try:
    import biom
    import h5py
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "No se ha podido importar biom/h5py. "
        "Activa el entorno con `conda activate empkg` o instala `biom-format` y `h5py`."
    ) from exc

# Verificación rápida de versiones
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"biom    {biom.__version__}")
print(f"h5py    {h5py.__version__}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 120)

pandas  3.0.3
numpy   2.4.6
biom    2.1.17
h5py    3.16.0


## 3. Rutas del proyecto

Definimos las rutas con `pathlib.Path` para que el notebook funcione tanto si
se ejecuta desde la raíz del repositorio como desde la carpeta `notebooks/`.

La lógica es simple: si el directorio de trabajo actual se llama `notebooks/`,
subimos un nivel para obtener la raíz del proyecto. En cualquier otro caso,
asumimos que ya estamos en la raíz.

Esto evita rutas hardcodeadas como `../../data/...` que rompen en cuanto
cambias de entorno.

In [295]:
# Detectar raíz del proyecto de forma robusta
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

# Rutas de datos
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "emp" / "release1"

MAPPING_PATH = DATA_RAW / "mapping_files" / "emp_qiime_mapping_subset_2k.tsv"
BIOM_PATH    = DATA_RAW / "otu_tables" / "deblur" / "emp_deblur_90bp.subset_2k.rare_5000.biom"

# Verificación
print(f"Raíz del proyecto : {PROJECT_ROOT}")
print(f"Mapping file      : {MAPPING_PATH}")
print(f"BIOM file         : {BIOM_PATH}")

Raíz del proyecto : /home/oier/EMPKG
Mapping file      : /home/oier/EMPKG/data/raw/emp/release1/mapping_files/emp_qiime_mapping_subset_2k.tsv
BIOM file         : /home/oier/EMPKG/data/raw/emp/release1/otu_tables/deblur/emp_deblur_90bp.subset_2k.rare_5000.biom


## 4. Descarga automática de archivos

Si los archivos de datos no están presentes en las rutas definidas,
esta celda los descarga desde el FTP del EMP.

La función `download_if_missing` es intencionalmente simple:
- Comprueba si el archivo ya existe antes de descargar.
- Crea los directorios necesarios.
- Descarga con `requests` en modo streaming para no cargar todo en memoria.
- Muestra una barra de progreso con `tqdm` si el servidor informa del tamaño.
- Imprime el tamaño final del archivo.


In [296]:
BASE_URL = "https://ftp.microbio.me/emp/release1"

DOWNLOADS = {
    MAPPING_PATH: f"{BASE_URL}/mapping_files/emp_qiime_mapping_subset_2k.tsv",
    BIOM_PATH:    f"{BASE_URL}/otu_tables/deblur/emp_deblur_90bp.subset_2k.rare_5000.biom",
}

def download_if_missing(path: Path, url: str) -> None:
    """
    Descarga `url` en `path` si el archivo no existe todavía.
    No hace nada si el archivo ya existe y tiene tamaño > 0.
    """
    if path.exists() and path.stat().st_size > 0:
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"OK (ya existe, {size_mb:.1f} MB): {path.relative_to(PROJECT_ROOT)}")
        return

    print(f"Descargando: {url}")
    path.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))

    with path.open("wb") as f:
        with tqdm(total=total, unit="B", unit_scale=True, desc=path.name) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"Descarga completada: {path.relative_to(PROJECT_ROOT)} ({size_mb:.1f} MB)\n")


# Descargar lo que falte
for target_path, url in DOWNLOADS.items():
    download_if_missing(target_path, url)

OK (ya existe, 1.5 MB): data/raw/emp/release1/mapping_files/emp_qiime_mapping_subset_2k.tsv
OK (ya existe, 53.8 MB): data/raw/emp/release1/otu_tables/deblur/emp_deblur_90bp.subset_2k.rare_5000.biom


## 5. Comprobación inicial de archivos

Antes de abrir nada, verificamos que ambos archivos existen y tienen un tamaño
razonable. Es un paso sencillo pero útil: si algo falla más adelante, sabremos
que no es un problema de rutas o de descarga incompleta.

In [297]:
def file_info(path: Path) -> None:
    if not path.exists():
        print(f"NO ENCONTRADO: {path}")
        return

    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"OK  {size_mb:7.1f} MB  {path.resolve()}")


file_info(MAPPING_PATH)
file_info(BIOM_PATH)

OK      1.5 MB  /home/oier/EMPKG/data/raw/emp/release1/mapping_files/emp_qiime_mapping_subset_2k.tsv
OK     53.8 MB  /home/oier/EMPKG/data/raw/emp/release1/otu_tables/deblur/emp_deblur_90bp.subset_2k.rare_5000.biom


## 6. Cargar el BIOM con `biom.load_table`

`biom.load_table` es la función principal de la librería `biom-format` para
abrir archivos `.biom`. Devuelve un objeto de tipo `biom.Table`, que representa
internamente una **matriz dispersa** (no carga todos los valores en memoria).

Un archivo BIOM tiene dos dimensiones:

- **Observaciones** (`observation`): en este caso, ASVs. Cada ASV es una
  secuencia nucleotídica única detectada por Deblur.
- **Muestras** (`sample`): cada muestra es una toma de material ambiental
  procesada en laboratorio.

La celda `table` de la intersección entre un ASV y una muestra contiene el
**número de lecturas** de esa secuencia en esa muestra.
La gran mayoría de celdas son cero: la mayoría de ASVs no aparece en la mayoría
de muestras. Por eso se usa una representación dispersa.

In [298]:
# Cargar el BIOM. La cadena str() es necesaria porque biom-format
# no acepta objetos Path, solo strings.
table = biom.load_table(str(BIOM_PATH))

# Jupyter muestra un resumen compacto del objeto si lo evaluamos solo
table

155002 x 2000 <class 'biom.table.Table'> with 858173 nonzero entries (0% dense)

### Métricas básicas del objeto `biom.Table`

Inspeccionamos los atributos principales para entender qué tenemos.

In [299]:
n_obs, n_samples = table.shape

print(f"Observaciones (ASVs) : {n_obs:>10,}")
print(f"Muestras             : {n_samples:>10,}")
print()
print(f"Valores no nulos (nnz)    : {table.nnz:>10,}")
print(f"Densidad de la matriz     : {table.get_table_density():>10.4%}")
print()
print(f"ID de la tabla  : {table.table_id}")
print(f"Versión formato : {table.format_version}")
print(f"Tipo declarado  : {table.type}")

Observaciones (ASVs) :    155,002
Muestras             :      2,000

Valores no nulos (nnz)    :    858,173
Densidad de la matriz     :    0.2768%

ID de la tabla  : No Table ID
Versión formato : (2, 1)
Tipo declarado  : OTU table


## 7. Entender los ejes: samples vs observations

Un objeto `biom.Table` tiene dos ejes, y es importante no confundirlos:

| Eje             | Argumento         | Qué representa en este archivo     |
|-----------------|-------------------|------------------------------------|
| Muestras        | `axis="sample"`   | Las 2.000 tomas de material ambiental |
| Observaciones   | `axis="observation"` | Los 155.002 ASVs detectados por Deblur |

Para acceder a los IDs de cualquiera de los dos ejes se usa `table.ids(axis=...)`.

**Nota sobre la terminología:**
El BIOM puede declarar su tipo como `OTU table`, pero las observaciones de este
archivo son **ASVs** (Amplicon Sequence Variants), no OTUs. La diferencia es
importante:

- Un **OTU** agrupa secuencias con ≥97% de similitud. Es una categoría difusa.
- Un **ASV** es una secuencia exacta, sin agrupación. Es el estándar actual.

En Deblur, el ID de cada ASV *es* la propia secuencia nucleotídica. Por eso los
IDs de observación parecen cadenas de ADN y no nombres legibles.

In [300]:
# Extraer IDs de ambos ejes
sample_ids      = list(table.ids(axis="sample"))
observation_ids = list(table.ids(axis="observation"))

print(f"Número de muestras     : {len(sample_ids):,}")
print(f"Número de ASVs         : {len(observation_ids):,}")

Número de muestras     : 2,000
Número de ASVs         : 155,002


### IDs de muestra

Los IDs de muestra son cadenas alfanuméricas asignadas por el EMP.
No contienen información ambiental por sí solos — esa información
está en el mapping file TSV, vinculada por este mismo ID.

In [301]:
# Los IDs tienen un formato como 10317.000003428 o similar, que es {study_id}.{sample_number}
# Este mismo valor aparece en la columna #SampleID del mapping file.
print("Primeros 5 IDs de muestra:")
for sid in sample_ids[:5]:
    print(f"  {sid}")

Primeros 5 IDs de muestra:
  1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex
  1453.45796SDZ4.G4.Pnem.stom
  1039.L.Vermelha.SA
  1773.Thraupis.gauco3.lgi
  1453.45300SDZ4.D7.Pnem.stom


### IDs de observación (ASVs)

En Deblur, el ID de cada ASV es la secuencia nucleotídica completa (90 bp en
este archivo). Son cadenas largas de A, T, C, G que no resultan legibles
directamente, pero son únicas e inequívocas.

Recortamos la visualización a los primeros 30 caracteres para que sean manejables.

In [302]:
print("Primeros 5 IDs de ASV (recortados a 30 caracteres):")
for oid in observation_ids[:5]:
    print(f"  {oid[:30]}...  (longitud total: {len(oid)} bp)")

Primeros 5 IDs de ASV (recortados a 30 caracteres):
  TACGGAGGGTGCAAGCGTTAATCGGAATTA...  (longitud total: 90 bp)
  TACGGAGTGTGCAAGCGTTACTCGGAATCA...  (longitud total: 90 bp)
  TACGAGAGGTCCAAACGTTATTCGGAATTA...  (longitud total: 90 bp)
  TACGTAGGTGGCAAGCGTTATCCGGAATTA...  (longitud total: 90 bp)
  TACGGAAGGTCCGGGCGTTATCCGGATTTA...  (longitud total: 90 bp)


### ¿Hay metadatos de muestra dentro del BIOM?

En algunos archivos BIOM los metadatos de muestra van embebidos dentro del
propio archivo. En el EMP, la convención es mantenerlos separados en el
mapping file TSV. Vale la pena comprobarlo explícitamente para no asumir nada.

In [303]:
sample_meta = table.metadata(axis="sample")

if sample_meta is None or len(sample_meta) == 0:
    print("Los metadatos de muestra NO están en el BIOM.")
    print("→ Están en el mapping file TSV. Esto es lo esperado en EMP.")
else:
    print(f"El BIOM contiene metadatos de muestra.")
    print(f"Campos disponibles: {list(sample_meta[0].keys())}")

Los metadatos de muestra NO están en el BIOM.
→ Están en el mapping file TSV. Esto es lo esperado en EMP.


### Metadatos de observación: taxonomía

Las observaciones sí tienen metadatos embebidos en el BIOM: la taxonomía
de cada ASV en 7 niveles jerárquicos. Inspeccionamos la primera observación
como ejemplo.

In [304]:
first_obs_id = observation_ids[0]
first_obs_meta = table.metadata(id=first_obs_id, axis="observation")

print(f"ASV ID (completo) : {first_obs_id}")
print(f"\nCampos de metadatos disponibles: {list(first_obs_meta.keys())}")


niveles = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]


for nivel, valor in zip(niveles, first_obs_meta["taxonomy"]):
    # El formato Greengenes tiene prefijos tipo "k__", "p__", etc.
    # Primer carácter del nivel + __ + taxon
    print(f"  {nivel:<10} {valor}")

ASV ID (completo) : TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGCTGTTTAAGCTAGCTGTGAAAGCCCCGGGCTTAAC

Campos de metadatos disponibles: ['taxonomy']
  kingdom    k__Bacteria
  phylum     p__Proteobacteria
  class      c__Gammaproteobacteria
  order      o__Alteromonadales
  family     f__Alteromonadaceae
  genus      g__Marinimicrobium
  species    s__


## 8. Sumas y recuentos básicos

Antes de construir ninguna tabla, conviene entender qué información
podemos extraer del BIOM eje por eje.

Hay cuatro operaciones fundamentales:

| Operación                          | Método                                  | Resultado                                      |
|------------------------------------|-----------------------------------------|------------------------------------------------|
| Total de lecturas por muestra      | `table.sum(axis="sample")`              | Array de 2.000 valores (uno por muestra)       |
| Abundancia total de cada ASV       | `table.sum(axis="observation")`         | Array de 155.002 valores (uno por ASV)         |
| Nº de ASVs distintos por muestra   | `table.nonzero_counts(axis="sample")`   | Array de 2.000 valores                         |
| Nº de muestras donde aparece un ASV| `table.nonzero_counts(axis="observation")` | Array de 155.002 valores                    |

Todas estas operaciones trabajan sobre la matriz dispersa internamente,
sin necesidad de convertirla a formato denso. Son eficientes incluso
con matrices grandes.

In [305]:
# Las cuatro operaciones básicas
sample_sums          = table.sum(axis="sample")           # lecturas por muestra
observation_sums     = table.sum(axis="observation")      # abundancia por ASV
observed_asvs_per_sample      = table.nonzero_counts(axis="sample")       # ASVs distintos por muestra
prevalence_per_asv      = table.nonzero_counts(axis="observation")  # muestras donde aparece cada ASV

print(f"sample_sums       → shape: {sample_sums.shape},      dtype: {sample_sums.dtype}")
print(f"observation_sums  → shape: {observation_sums.shape}, dtype: {observation_sums.dtype}")
print(f"observed_asvs_per_sample   → shape: {observed_asvs_per_sample.shape},      dtype: {observed_asvs_per_sample.dtype}")
print(f"prevalence_per_asv   → shape: {prevalence_per_asv.shape}, dtype: {prevalence_per_asv.dtype}")

sample_sums       → shape: (2000,),      dtype: float64
observation_sums  → shape: (155002,), dtype: float64
observed_asvs_per_sample   → shape: (2000,),      dtype: int64
prevalence_per_asv   → shape: (155002,), dtype: int64


### 8.1 Total de lecturas por muestra (`sample_sums`)

Este archivo está rarefaccionado a 5.000 lecturas por muestra (`rare_5000`).
Eso significa que todas las muestras deberían tener exactamente 5.000 lecturas
totales. La rarefacción es una técnica de normalización que submuestrea cada
muestra al mismo número de lecturas para que sean comparables entre sí.

Si alguna muestra tenía menos de 5.000 lecturas originalmente, fue eliminada
del dataset rarefaccionado.

In [306]:
sample_sums_series = pd.Series(sample_sums, index=table.ids(axis="sample"), name="total_reads")

print("Estadísticas de lecturas totales por muestra:")
print(sample_sums_series.describe().round(1))
print()
print(f"¿Todas las muestras tienen exactamente 5.000 lecturas? "
      f"{'Sí' if sample_sums_series.nunique() == 1 else 'No'}")
print(f"Valor mínimo: {sample_sums_series.min():.0f}")
print(f"Valor máximo: {sample_sums_series.max():.0f}")

Estadísticas de lecturas totales por muestra:
count    2000.0
mean     5000.0
std         0.0
min      5000.0
25%      5000.0
50%      5000.0
75%      5000.0
max      5000.0
Name: total_reads, dtype: float64

¿Todas las muestras tienen exactamente 5.000 lecturas? Sí
Valor mínimo: 5000
Valor máximo: 5000


### 8.2 Abundancia total de cada ASV (`observation_sums`)

La abundancia total de un ASV es la suma de sus lecturas en todas las muestras
donde aparece. Nos dice qué ASVs son dominantes en el dataset global y cuáles
son raros.

En microbiomía es habitual encontrar una distribución muy sesgada: unos pocos
ASVs son muy abundantes y la mayoría son raros (aparecen en pocas muestras y
con pocas lecturas). Esto se llama distribución de ley de potencias o
distribución de cola larga.

In [307]:
obs_sums_series = pd.Series(observation_sums, index=table.ids(axis="observation"), name="total_abundance")

print("Estadísticas de abundancia total por ASV:")
print(obs_sums_series.describe().round(1))
print()
print(f"ASVs con abundancia total = 1  (singletons): {(obs_sums_series == 1).sum():,}")
print(f"ASVs con abundancia total ≤ 10             : {(obs_sums_series <= 10).sum():,}")
print(f"ASVs con abundancia total > 1.000          : {(obs_sums_series > 1_000).sum():,}")
print()
print("Los 10 ASVs más abundantes (ID recortado):")
top10 = obs_sums_series.nlargest(10)
for asv_id, abundance in top10.items():
    print(f"  {asv_id[:30]}...  →  {abundance:,.0f} lecturas")

Estadísticas de abundancia total por ASV:
count    155002.0
mean         64.5
std        1593.9
min           1.0
25%           1.0
50%           4.0
75%          12.0
max      526444.0
Name: total_abundance, dtype: float64

ASVs con abundancia total = 1  (singletons): 38,892
ASVs con abundancia total ≤ 10             : 112,306
ASVs con abundancia total > 1.000          : 1,325

Los 10 ASVs más abundantes (ID recortado):
  TACAGAGGATGCAAGCGTTATCCGGAATGA...  →  526,444 lecturas
  TACGTAGGGCGCAAGCGTTATCCGGAATTA...  →  108,473 lecturas
  TACGTAGGTCCCGAGCGTTGTCCGGATTTA...  →  94,816 lecturas
  TACGGAGGGGGCTAGCGTTGTTCGGAATTA...  →  87,411 lecturas
  TACGGAGGGTGCAAGCGTTAATCGGAATTA...  →  77,856 lecturas
  TACGTAGGTGGCAAGCGTTATCCGGAATTA...  →  76,020 lecturas
  TACGGAGGGTGCAAGCGTTAATCGGAATTA...  →  72,346 lecturas
  TACAGAGGGTGCAAGCGTTAATCGGAATTA...  →  63,775 lecturas
  TACGGAGGGTGCGAGCGTTAATCGGAATGA...  →  63,734 lecturas
  GACAGAGGATGCAAGCGTTATCCGGAATGA...  →  57,648 lecturas


### 8.3 Número de ASVs distintos por muestra (`observed_asvs_per_sample`)

Este recuento nos dice cuántos ASVs diferentes se detectaron en cada muestra,
independientemente de su abundancia. Es una medida sencilla de **riqueza
microbiana** (diversidad alfa en su forma más básica).

Muestras con más ASVs distintos tienen una comunidad microbiana más diversa,
al menos en términos de riqueza observada.

In [308]:
observed_asvs_per_sample_series = pd.Series(
    observed_asvs_per_sample,
    index=table.ids(axis="sample"),
    name="observed_asvs",
)

print("Estadísticas de ASVs observados por muestra:")
print(observed_asvs_per_sample_series.describe().round(1))
print()
print(f"Muestra con MENOS ASVs distintos : {observed_asvs_per_sample_series.min():.0f}")
print(f"Muestra con MÁS ASVs distintos   : {observed_asvs_per_sample_series.max():.0f}")
print(f"Mediana                          : {observed_asvs_per_sample_series.median():.0f}")

Estadísticas de ASVs observados por muestra:
count    2000.0
mean      429.1
std       465.3
min         1.0
25%        94.0
50%       245.5
75%       611.0
max      2438.0
Name: observed_asvs, dtype: float64

Muestra con MENOS ASVs distintos : 1
Muestra con MÁS ASVs distintos   : 2438
Mediana                          : 246


### 8.4 Número de muestras donde aparece cada ASV (`prevalence_per_asv`)

Este recuento nos dice en cuántas muestras aparece cada ASV. Un ASV que aparece en muchas muestras puede considerarse candidato a taxón generalista o ampliamente distribuido, aunque haría falta analizar el contexto ambiental para confirmarlo.

La gran mayoría de ASVs aparecen en muy pocas muestras. Esto refleja que
la diversidad microbiana tiene una estructura de cola larga: muchos taxones
raros y pocos taxones dominantes.

In [309]:
prevalence_per_asv_series = pd.Series(
    prevalence_per_asv,
    index=table.ids(axis="observation"),
    name="n_samples",
)

print("Estadísticas de muestras por ASV:")
print(prevalence_per_asv_series.describe().round(1))
print()

# Distribución por rangos: muestra la estructura de cola larga
rangos = [1, 2, 5, 10, 50, 100, 500]
print("Distribución acumulada (ASVs que aparecen en ≤ N muestras):")
for n in rangos:
    cuenta = (prevalence_per_asv_series <= n).sum()
    pct = cuenta / len(prevalence_per_asv_series) * 100
    print(f"  ≤ {n:>4} muestras : {cuenta:>8,} ASVs  ({pct:.1f}%)")

Estadísticas de muestras por ASV:
count    155002.0
mean          5.5
std          14.6
min           1.0
25%           1.0
50%           2.0
75%           4.0
max         614.0
Name: n_samples, dtype: float64

Distribución acumulada (ASVs que aparecen en ≤ N muestras):
  ≤    1 muestras :   66,338 ASVs  (42.8%)
  ≤    2 muestras :   94,221 ASVs  (60.8%)
  ≤    5 muestras :  123,953 ASVs  (80.0%)
  ≤   10 muestras :  138,432 ASVs  (89.3%)
  ≤   50 muestras :  152,689 ASVs  (98.5%)
  ≤  100 muestras :  154,361 ASVs  (99.6%)
  ≤  500 muestras :  154,998 ASVs  (100.0%)


## 9. Construir la tabla `sample_stats`

Con las operaciones de la sección anterior ya tenemos todo lo necesario para
construir una tabla resumen con una fila por muestra y tres columnas:

| Columna               | Origen                                      | Significado                              |
|-----------------------|---------------------------------------------|------------------------------------------|
| `sample_id`           | `table.ids(axis="sample")`                  | Identificador único de la muestra        |
| `biom_total_reads`    | `table.sum(axis="sample")`                  | Total de lecturas (debería ser 5.000)    |
| `biom_observed_asvs`  | `table.nonzero_counts(axis="sample")`       | Número de ASVs distintos en la muestra  |

Esta tabla es el **puente entre el BIOM y el mapping file**: la construimos
desde el BIOM y luego la unimos con los metadatos ambientales del TSV por `sample_id`.

Todavía no es `sample_table.csv` completo — eso lo haremos en la sección 14
cuando crucemos con el mapping file. Aquí construimos solo la parte que viene
del BIOM.

In [310]:
sample_stats = pd.DataFrame({
    "sample_id":          table.ids(axis="sample"),
    "biom_total_reads":   table.sum(axis="sample"),
    "biom_observed_asvs": table.nonzero_counts(axis="sample"),
}).set_index("sample_id")

print(f"Shape: {sample_stats.shape}")
print()
sample_stats.head()

Shape: (2000, 2)



,biom_total_reads,biom_observed_asvs
sample_id,,
1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex,5000.0,304
1453.45796SDZ4.G4.Pnem.stom,5000.0,117
1039.L.Vermelha.SA,5000.0,788
1773.Thraupis.gauco3.lgi,5000.0,189
1453.45300SDZ4.D7.Pnem.stom,5000.0,232


### Verificar la rarefacción

Comprobamos formalmente que todas las muestras tienen exactamente 5.000 lecturas.
Si el archivo estuviera corrupto o fuera una versión distinta de la esperada,
esta comprobación lo detectaría.

In [311]:
reads_unico = sample_stats["biom_total_reads"].unique()

if len(reads_unico) == 1 and reads_unico[0] == 5000.0:
    print("OK: todas las muestras tienen exactamente 5.000 lecturas.")
else:
    print(f"AVISO: hay variación en el número de lecturas.")
    print(f"Valores únicos encontrados: {reads_unico[:10]}")

OK: todas las muestras tienen exactamente 5.000 lecturas.


### Estadísticas descriptivas de `sample_stats`

Examinamos la distribución de ASVs observados por muestra.
Esta es la primera descripción cuantitativa de la diversidad microbiana
del subset de 2.000 muestras.

In [312]:
print("Estadísticas de biom_observed_asvs:")
print(sample_stats["biom_observed_asvs"].describe().round(1))
print()

# Percentiles adicionales para entender mejor la distribución
percentiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
print("Percentiles adicionales:")
for p in percentiles:
    valor = sample_stats["biom_observed_asvs"].quantile(p)
    print(f"  p{int(p*100):>3}  →  {valor:.0f} ASVs")

Estadísticas de biom_observed_asvs:
count    2000.0
mean      429.1
std       465.3
min         1.0
25%        94.0
50%       245.5
75%       611.0
max      2438.0
Name: biom_observed_asvs, dtype: float64

Percentiles adicionales:
  p  5  →  15 ASVs
  p 10  →  27 ASVs
  p 25  →  94 ASVs
  p 50  →  246 ASVs
  p 75  →  611 ASVs
  p 90  →  1162 ASVs
  p 95  →  1411 ASVs


### Identificar muestras en los extremos de diversidad

Las muestras con muy pocos o muy muchos ASVs son interesantes desde el punto
de vista ecológico. Veamos cuáles son antes de cruzar con el mapping file,
para saber a qué entorno corresponden cuando lleguemos a la sección 14.

In [313]:
n = 5  # cuántas muestras mostrar en cada extremo

menos_diversas = sample_stats["biom_observed_asvs"].nsmallest(n)
mas_diversas   = sample_stats["biom_observed_asvs"].nlargest(n)

print(f"Las {n} muestras con MENOS ASVs distintos:")
print(menos_diversas.to_frame().to_string())
print()
print(f"Las {n} muestras con MÁS ASVs distintos:")
print(mas_diversas.to_frame().to_string())

Las 5 muestras con MENOS ASVs distintos:
                                                                 biom_observed_asvs
sample_id                                                                          
2382.MU002.C3.HA.5.699.leav.9.12.lane8.NoIndex.L008.sequences                     1
2382.SH008.C6.HA.4.745.gp.9.12.lane8.NoIndex.L008.sequences                       2
2382.MU002.C3.HA.4.76.r1.leav.6.11.lane8.NoIndex.L008.sequences                   2
2382.SH007.C6.RH.4.719.leav.9.12.lane8.NoIndex.L008.sequences                     2
2382.SH004.C1.RH.2.609.leav.9.12.lane8.NoIndex.L008.sequences                     2

Las 5 muestras con MÁS ASVs distintos:
                                                   biom_observed_asvs
sample_id                                                            
1883.2010.21.Crump.Artic.LTREB.main.lane3.NoIndex                2438
1521.34.W.s.6.1.sequences                                        2390
1521.76.W.s.6.1.sequences                          

### Convertir columnas a entero

Las columnas vienen como `float64` por cómo BIOM almacena internamente los
valores. Como sabemos que son recuentos enteros, las convertimos a `int64`
para que la tabla sea más limpia y ocupe menos memoria.

In [314]:
sample_stats = sample_stats.astype({
    "biom_total_reads":   "int64",
    "biom_observed_asvs": "int64",
})

print("Tipos de columna tras la conversión:")
print(sample_stats.dtypes)
print()
print("Vista final de sample_stats:")
sample_stats.head(10)

Tipos de columna tras la conversión:
biom_total_reads      int64
biom_observed_asvs    int64
dtype: object

Vista final de sample_stats:


,biom_total_reads,biom_observed_asvs
sample_id,,
1883.2008.269.Crump.Artic.LTREB.main.lane2.NoIndex,5000,304
1453.45796SDZ4.G4.Pnem.stom,5000,117
1039.L.Vermelha.SA,5000,788
1773.Thraupis.gauco3.lgi,5000,189
1453.45300SDZ4.D7.Pnem.stom,5000,232
2382.SH004.C1.RH.1.605.gp.9.12.lane8.NoIndex.L008.sequences,5000,5
1774.139.Oral.Puer,5000,89
1627.LC,5000,335
722.TRRsed1.5.s.6.1.sequence,5000,668


## 10. Explorar la taxonomía de las observaciones

Cada ASV del BIOM tiene asociados metadatos de observación con su clasificación taxonómica en 7 niveles jerárquicos. Esta taxonomía fue asignada automáticamente
por Deblur comparando cada secuencia contra una base de datos de referencia (Greengenes 13.8 en el caso del EMP Release 1).

La jerarquía taxonómica estándar es:
Reino (kingdom)
└── Filo (phylum)

└── Clase (class)

└── Orden (order)

└── Familia (family)

└── Género (genus)

└── Especie (species)

En la notación Greengenes cada nivel lleva un prefijo de dos caracteres:

| Nivel   | Prefijo | Ejemplo                  |
|---------|---------|--------------------------|
| kingdom | `k__`   | `k__Bacteria`            |
| phylum  | `p__`   | `p__Proteobacteria`      |
| class   | `c__`   | `c__Gammaproteobacteria` |
| order   | `o__`   | `o__Pseudomonadales`     |
| family  | `f__`   | `f__Pseudomonadaceae`    |
| genus   | `g__`   | `g__Pseudomonas`         |
| species | `s__`   | `s__` *(a menudo vacío)* |

Cuanto más bajo es el nivel, más probable es que esté vacío: la clasificación automática pierde resolución en los niveles más finos. Tener el filo o la clase asignados es habitual; tener la especie es la excepción.

In [315]:
# Inspeccionamos los metadatos de la última observación como ejemplo
last_obs_id   = observation_ids[-1]
last_obs_meta = table.metadata(id=last_obs_id, axis="observation")

print(f"Campos disponibles en metadatos de observación:")
print(f"  {list(last_obs_meta.keys())}")
print()
print(f"Taxonomía completa del último ASV:")
for entrada in last_obs_meta["taxonomy"]:
    print(f"  {entrada}")

Campos disponibles en metadatos de observación:
  ['taxonomy']

Taxonomía completa del último ASV:
  k__Bacteria
  p__Firmicutes
  c__Clostridia
  o__Clostridiales
  f__[Tissierellaceae]
  g__Helcococcus
  s__


### Construir `taxonomy_preview`

Construimos una tabla con los primeros 20 ASVs del BIOM donde cada nivel
taxonómico es una columna separada, sin los prefijos Greengenes.

Esto nos permite ver de un vistazo:
- Qué reinos están representados.
- A qué nivel llega típicamente la clasificación.
- Qué aspecto tienen los ASVs sin clasificación en niveles finos.

In [316]:
TAX_LEVELS = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
N_PREVIEW  = 20

tax_rows = []

for asv_id in observation_ids[:N_PREVIEW]:
    meta    = table.metadata(id=asv_id, axis="observation")
    tax_raw = meta.get("taxonomy", [])

    # Eliminar el prefijo de dos caracteres + "__" de cada nivel
    # "k__Bacteria" → "Bacteria",  "g__" → "" (vacío)
    tax_clean = [t[3:] if len(t) > 3 else "" for t in tax_raw]

    # Rellenar con cadena vacía si hay menos de 7 niveles
    tax_clean += [""] * (len(TAX_LEVELS) - len(tax_clean))

    row = {"asv_id": asv_id[:35] + "..."}
    row.update(dict(zip(TAX_LEVELS, tax_clean)))
    tax_rows.append(row)

taxonomy_preview = pd.DataFrame(tax_rows).set_index("asv_id")

taxonomy_preview

,kingdom,phylum,class,order,family,genus,species
asv_id,,,,,,,
TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGG...,Bacteria,Proteobacteria,Gammaproteobacteria,Alteromonadales,Alteromonadaceae,Marinimicrobium,
TACGGAGTGTGCAAGCGTTACTCGGAATCACTGGG...,Bacteria,Planctomycetes,028H05-P-BN-P5,,,,
TACGAGAGGTCCAAACGTTATTCGGAATTACTGGG...,Bacteria,Planctomycetes,Planctomycetia,Pirellulales,Pirellulaceae,,
TACGTAGGTGGCAAGCGTTATCCGGAATTACTGGG...,Bacteria,Firmicutes,,,,,
TACGGAAGGTCCGGGCGTTATCCGGATTTATTGGG...,Bacteria,Bacteroidetes,Bacteroidia,Bacteroidales,[Paraprevotellaceae],,
TACGAGGGGTGCAAACGTTATTCGGAATGATTGGG...,Bacteria,TM6,SJA-4,,,,
TACGGAGGGTGCAAGCGTTATCCGGAATCACTGGG...,Bacteria,,,,,,
TACAGACGGTCCAAACGTTATTCGGAATTACTGGG...,Bacteria,Planctomycetes,Planctomycetia,Pirellulales,Pirellulaceae,Pirellula,
TACGAGGGGGGCTAGCGTTGTTCGGATTTACTGGG...,Bacteria,,,,,,


### Cobertura taxonómica por nivel

Una pregunta clave para el Knowledge Graph es: ¿a qué nivel taxonómico
tenemos clasificación fiable para la mayoría de ASVs?

Calculamos la cobertura usando una muestra aleatoria reproducible de 5.000 ASVs para no recorrer los 155.002.

In [ ]:
rng = np.random.default_rng(seed=42)

N_SAMPLE = 5_000  # subconjunto rápido para exploración inicial
selected_obs_ids = rng.choice(observation_ids, size=N_SAMPLE, replace=False)

tax_sample_rows = []

for asv_id in selected_obs_ids:
    meta = table.metadata(id=asv_id, axis="observation")
    tax_raw = meta.get("taxonomy", [])

    tax_clean = [t[3:] if len(t) > 3 else "" for t in tax_raw]
    tax_clean += [""] * (len(TAX_LEVELS) - len(tax_clean))

    tax_sample_rows.append(dict(zip(TAX_LEVELS, tax_clean)))

tax_sample_df = pd.DataFrame(tax_sample_rows, index=selected_obs_ids)

print(f"Cobertura taxonómica en una muestra aleatoria de {N_SAMPLE:,} ASVs:\n")
print(f"  {'Nivel':<12}  {'Con asignación':>15}  {'Vacíos':>10}  {'Cobertura':>10}")
print(f"  {'-'*52}")

for nivel in TAX_LEVELS:
    con_asig = (tax_sample_df[nivel] != "").sum()
    vacios   = N_SAMPLE - con_asig
    cobertura = con_asig / N_SAMPLE * 100
    print(f"  {nivel:<12}  {con_asig:>15,}  {vacios:>10,}  {cobertura:>9.1f}%")

Cobertura taxonómica en los primeros 5,000 ASVs:

  Nivel          Con asignación      Vacíos   Cobertura
  ----------------------------------------------------
  kingdom                 5,000           0      100.0%
  phylum                  4,396         604       87.9%
  class                   3,814       1,186       76.3%
  order                   2,977       2,023       59.5%
  family                  1,769       3,231       35.4%
  genus                     696       4,304       13.9%
  species                    85       4,915        1.7%


### Distribución de filos (phylum)

El filo es el nivel taxonómico con mejor equilibrio entre resolución biológica y cobertura en los datos. Veamos qué filos dominan en el dataset.

Usamos la misma muestra de 5.000 ASVs ponderada por abundancia total, para reflejar no solo cuántos ASVs hay de cada filo sino cuántas lecturas representan.

In [318]:
# Abundancia total de cada ASV (calculada en sección 8)
obs_sums_series = pd.Series(
    table.sum(axis="observation"),
    index=table.ids(axis="observation"),
    name="total_abundance",
)

# Añadir abundancia a la tabla de taxonomía de muestra
tax_sample_df.index = observation_ids[:N_SAMPLE]
tax_sample_df["total_abundance"] = obs_sums_series.iloc[:N_SAMPLE].values

# Distribución por filo: número de ASVs y abundancia total
phylum_stats = (
    tax_sample_df
    .groupby("phylum", dropna=False)
    .agg(
        n_asvs=("phylum", "count"),
        total_reads=("total_abundance", "sum"),
    )
    .sort_values("total_reads", ascending=False)
)

phylum_stats["pct_reads"] = (phylum_stats["total_reads"] / phylum_stats["total_reads"].sum() * 100).round(1)

print(f"Distribución por filo (top 15, muestra de {N_SAMPLE:,} ASVs):\n")
print(phylum_stats.head(15).to_string())

Distribución por filo (top 15, muestra de 5,000 ASVs):

                  n_asvs  total_reads  pct_reads
phylum                                          
Firmicutes           372     106526.0       29.0
Proteobacteria      1280      79703.0       21.7
Bacteroidetes        463      26855.0        7.3
Chloroflexi          265      23878.0        6.5
Planctomycetes       359      23697.0        6.5
                     604      17724.0        4.8
OP3                   62       9964.0        2.7
Actinobacteria       242       9950.0        2.7
Gemmatimonadetes      65       8815.0        2.4
WS3                   40       8274.0        2.3
Acidobacteria        227       8057.0        2.2
Tenericutes           30       7003.0        1.9
Cyanobacteria        128       6987.0        1.9
OD1                   96       5758.0        1.6
Caldithrix             4       4077.0        1.1


## 11. Inspeccionar la estructura HDF5 con `h5py`

El formato BIOM 2.x está construido sobre **HDF5** (Hierarchical Data Format 5),
un formato binario diseñado para almacenar grandes volúmenes de datos científicos
de forma eficiente.

Hasta ahora hemos trabajado con el BIOM a través de la capa de abstracción de
`biom-format`, que nos oculta los detalles de almacenamiento. En esta sección
bajamos un nivel y abrimos el archivo directamente con `h5py` para ver cómo
está organizado internamente.

Esto es útil por dos razones:

1. **Comprensión:** entender por qué `biom-format` es eficiente y qué significa
   que la matriz sea "dispersa".
2. **Diagnóstico:** si `biom.load_table` falla alguna vez, saber leer el HDF5
   directamente permite inspeccionar el archivo igualmente.

### Conceptos clave de HDF5

HDF5 organiza los datos en torno a dos objetos principales:

- **Grupo (`Group`):** equivalente a una carpeta. Puede contener otros grupos
  o datasets.
- **Dataset:** array multidimensional de datos (números, cadenas, etc.).
  Equivalente a un fichero dentro de una carpeta.

Adicionalmente, tanto los grupos como los datasets pueden tener **atributos**:
pares clave-valor que almacenan metadatos sobre el contenido
(versión del formato, tipo de tabla, fecha de creación...).

La estructura de un BIOM 2.1 tiene siempre esta forma:
/                          ← raíz

├── observation/           ← grupo: eje de ASVs

│   ├── ids                ← dataset: array de IDs de ASV

│   ├── matrix/            ← grupo: datos de la matriz dispersa (lado ASV)

│   │   ├── indptr

│   │   ├── indices

│   │   └── data

│   └── metadata/          ← grupo: metadatos de cada ASV

│       └── taxonomy       ← dataset: taxonomía en 7 niveles

└── sample/                ← grupo: eje de muestras

├── ids                ← dataset: array de IDs de muestra

├── matrix/            ← grupo: datos de la matriz dispersa (lado muestra)

│   ├── indptr

│   ├── indices

│   └── data

└── metadata/          ← grupo: metadatos de cada muestra (vacío en EMP)

La matriz de abundancias se almacena en formato **CSC**
(Compressed Sparse Column): en lugar de guardar todos los ceros, solo se guardan los valores no nulos y su posición. De ahí la eficiencia.

### Atributos del archivo

Los atributos de la raíz del HDF5 contienen metadatos globales del archivo:
versión del formato, tipo de tabla, fechas, etc.

In [319]:
with h5py.File(BIOM_PATH, "r") as f:
    print("Atributos del archivo HDF5:\n")
    for key, value in f.attrs.items():
        print(f"  {key:<30} → {value}")

Atributos del archivo HDF5:

  creation-date                  → 2016-09-28T08:57:11.660674
  format-url                     → http://biom-format.org
  format-version                 → [2 1]
  generated-by                   → QIIME 1.9.1
  id                             → No Table ID
  nnz                            → 858173
  shape                          → [155002   2000]
  type                           → OTU table


### Estructura jerárquica completa

Recorremos todos los grupos y datasets del archivo con `visititems`,
que funciona como un `os.walk` para HDF5.

In [320]:
def mostrar_estructura_hdf5(biom_path: Path) -> None:
    """Recorre el HDF5 e imprime grupos y datasets con su forma."""

    def visitor(name, obj):
        profundidad = name.count("/")
        sangria     = "  " * profundidad
        tipo        = "GRUPO  " if isinstance(obj, h5py.Group) else "DATASET"
        forma       = f"  shape={obj.shape}  dtype={obj.dtype}" if isinstance(obj, h5py.Dataset) else ""
        print(f"  {sangria}{tipo}  /{name}{forma}")

    with h5py.File(biom_path, "r") as f:
        print("Estructura del archivo HDF5:\n")
        f.visititems(visitor)

mostrar_estructura_hdf5(BIOM_PATH)

Estructura del archivo HDF5:

  GRUPO    /observation
    GRUPO    /observation/group-metadata
    DATASET  /observation/ids  shape=(155002,)  dtype=object
    GRUPO    /observation/matrix
      DATASET  /observation/matrix/data  shape=(858173,)  dtype=float64
      DATASET  /observation/matrix/indices  shape=(858173,)  dtype=int32
      DATASET  /observation/matrix/indptr  shape=(155003,)  dtype=int32
    GRUPO    /observation/metadata
      DATASET  /observation/metadata/taxonomy  shape=(155002, 7)  dtype=object
  GRUPO    /sample
    GRUPO    /sample/group-metadata
    DATASET  /sample/ids  shape=(2000,)  dtype=object
    GRUPO    /sample/matrix
      DATASET  /sample/matrix/data  shape=(858173,)  dtype=float64
      DATASET  /sample/matrix/indices  shape=(858173,)  dtype=int32
      DATASET  /sample/matrix/indptr  shape=(2001,)  dtype=int32
    GRUPO    /sample/metadata


### Inspeccionar `/observation/ids`

Este dataset contiene los IDs de todos los ASVs, en el mismo orden
en que aparecen en la matriz.

In [321]:
with h5py.File(BIOM_PATH, "r") as f:
    obs_ids_hdf5 = f["observation/ids"][:]

print(f"Tipo del dataset  : {obs_ids_hdf5.dtype}")
print(f"Shape             : {obs_ids_hdf5.shape}")
print(f"Primeros 3 IDs    :")
for oid in obs_ids_hdf5[:3]:
    # En HDF5 las cadenas pueden venir como bytes; decodificamos si hace falta
    decoded = oid.decode("utf-8") if isinstance(oid, bytes) else oid
    print(f"  {decoded[:40]}...")

Tipo del dataset  : object
Shape             : (155002,)
Primeros 3 IDs    :
  TACGGAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAA...
  TACGGAGTGTGCAAGCGTTACTCGGAATCACTGGGCATAA...
  TACGAGAGGTCCAAACGTTATTCGGAATTACTGGGCTTAA...


### Inspeccionar `/observation/metadata/taxonomy`

La taxonomía está almacenada como un dataset 2D: una fila por ASV,
una columna por nivel taxonómico (7 columnas).

In [322]:
with h5py.File(BIOM_PATH, "r") as f:
    taxonomy_hdf5 = f["observation/metadata/taxonomy"]

    print(f"Shape del dataset taxonomy : {taxonomy_hdf5.shape}")
    print(f"Dtype                      : {taxonomy_hdf5.dtype}")
    print()
    print("Primeras 3 filas (un ASV por fila, 7 niveles por columna):")
    for i, row in enumerate(taxonomy_hdf5[:3]):
        decoded = [v.decode("utf-8") if isinstance(v, bytes) else v for v in row]
        print(f"  ASV {i}: {decoded}")

Shape del dataset taxonomy : (155002, 7)
Dtype                      : object

Primeras 3 filas (un ASV por fila, 7 niveles por columna):
  ASV 0: ['k__Bacteria', 'p__Proteobacteria', 'c__Gammaproteobacteria', 'o__Alteromonadales', 'f__Alteromonadaceae', 'g__Marinimicrobium', 's__']
  ASV 1: ['k__Bacteria', 'p__Planctomycetes', 'c__028H05-P-BN-P5', 'o__', 'f__', 'g__', 's__']
  ASV 2: ['k__Bacteria', 'p__Planctomycetes', 'c__Planctomycetia', 'o__Pirellulales', 'f__Pirellulaceae', 'g__', 's__']


### Inspeccionar la matriz dispersa

BIOM 2.x almacena la matriz de abundancias de forma dispersa, pero mantiene dos orientaciones:

- `observation/matrix`: representación orientada a observaciones. Permite acceder eficientemente a vectores de ASVs.
- `sample/matrix`: representación orientada a muestras. Permite acceder eficientemente a vectores de muestras.

Ambas representaciones contienen los mismos valores no nulos, pero organizados de forma distinta para acelerar operaciones sobre cada eje.

Cada representación usa tres arrays:

- `data`: valores no cero.
- `indices`: posiciones asociadas a esos valores.
- `indptr`: punteros que delimitan cada vector del eje correspondiente.

In [323]:
with h5py.File(BIOM_PATH, "r") as f:
    for group_name in ["observation/matrix", "sample/matrix"]:
        print(f"\n{group_name}")
        print("-" * len(group_name))

        data = f[f"{group_name}/data"]
        indices = f[f"{group_name}/indices"]
        indptr = f[f"{group_name}/indptr"]

        print(f"data    shape={data.shape}    dtype={data.dtype}")
        print(f"indices shape={indices.shape} dtype={indices.dtype}")
        print(f"indptr  shape={indptr.shape}  dtype={indptr.dtype}")

        print(f"Primeros valores de data   : {data[:10]}")
        print(f"Primeros valores de indices: {indices[:10]}")
        print(f"Primeros valores de indptr : {indptr[:10]}")

print("\nComprobaciones:")
print(f"  table.nnz = {table.nnz:,}")
print("  observation/matrix y sample/matrix almacenan los mismos valores no cero,")
print("  pero organizados según ejes distintos.")


observation/matrix
------------------
data    shape=(858173,)    dtype=float64
indices shape=(858173,) dtype=int32
indptr  shape=(155003,)  dtype=int32
Primeros valores de data   : [ 1. 18.  1.  2.  1.  2.  1.  1.  1.  1.]
Primeros valores de indices: [ 620  682 1905 1933  359  802 1413 1616  885 1370]
Primeros valores de indptr : [ 0  2  4  8 10 11 13 14 16 17]

sample/matrix
-------------
data    shape=(858173,)    dtype=float64
indices shape=(858173,) dtype=int32
indptr  shape=(2001,)  dtype=int32
Primeros valores de data   : [ 3.  9.  3.  2.  3.  1.  2. 38.  2. 11.]
Primeros valores de indices: [ 399  624  873 1126 1858 2001 2355 2870 3163 3364]
Primeros valores de indptr : [   0  304  421 1209 1398 1630 1635 1724 2059 2727]

Comprobaciones:
  table.nnz = 858,173
  observation/matrix y sample/matrix almacenan los mismos valores no cero,
  pero organizados según ejes distintos.


## 12. Cargar el mapping file TSV

El mapping file es un archivo TSV (valores separados por tabuladores) donde
cada fila es una muestra y cada columna es un campo de metadatos ambientales.

Es el complemento indispensable del BIOM: mientras el BIOM contiene
**quién está** (qué ASVs y en qué abundancia), el mapping file contiene
**de dónde viene** cada muestra (bioma, coordenadas, pH, temperatura...).

### Decisiones de carga

Hay dos decisiones importantes al cargar este archivo con pandas:

**1. `dtype=str`**
Forzamos que todas las columnas se lean como texto, sin que pandas intente inferir tipos automáticamente. Esto evita que columnas numéricas con valores ausentes mezclen `float` y `str`, o que IDs numéricos como `10317` se lean
como enteros y luego no coincidan con los IDs del BIOM.

**2. `keep_default_na=False` y `na_filter=False`**
Por defecto, pandas convierte automáticamente cadenas como `""`, `"NA"`, `"NaN"`, `"None"` o `"null"` a `NaN` antes de que podamos verlos.
Con estas dos opciones desactivamos ese comportamiento y controlamos
nosotros qué se trata como ausente.

Esto es importante porque en el mapping file del EMP los valores ausentes usan marcadores no estándar como `"Missing: Not provided"` o `"unknown"`, que pandas no reconocería por defecto y dejaría como texto sin normalizar.
Ya investigamos esto en detalle en `inspect_missing_values.py`.

In [324]:
mapping = pd.read_csv(
    MAPPING_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    na_filter=False,
)

print(f"Shape: {mapping.shape}")
print(f"  → {mapping.shape[0]:,} muestras  ×  {mapping.shape[1]:,} columnas")
print()
print("Primeras 10 columnas:")
print(f"  {mapping.columns[:10].tolist()}")
print()
print("Últimas 10 columnas:")
print(f"  {mapping.columns[-10:].tolist()}")

Shape: (2000, 76)
  → 2,000 muestras  ×  76 columnas

Primeras 10 columnas:
  ['#SampleID', 'BarcodeSequence', 'LinkerPrimerSequence', 'Description', 'host_subject_id', 'study_id', 'title', 'principal_investigator', 'doi', 'ebi_accession']

Últimas 10 columnas:
  ['adiv_shannon', 'adiv_faith_pd', 'temperature_deg_c', 'ph', 'salinity_psu', 'oxygen_mg_per_l', 'phosphate_umol_per_l', 'ammonium_umol_per_l', 'nitrate_umol_per_l', 'sulfate_umol_per_l']


### Columnas relevantes para el Knowledge Graph

De las 76 columnas disponibles, no todas son útiles para el KG.
Seleccionamos las más relevantes agrupadas por categoría para
inspeccionarlas con más detalle.

In [325]:
COLS_IDENTIDAD = ["#SampleID", "study_id"]

COLS_EMPO = ["empo_0", "empo_1", "empo_2", "empo_3"]

COLS_ENVO = [
    "env_biome",
    "env_feature",
    "env_material",
    "envo_biome_0",
    "envo_biome_1",
    "envo_biome_2",
    "envo_biome_3",
]

COLS_GEO = [
    "country",
    "latitude_deg",
    "longitude_deg",
    "depth_m",
    "altitude_m",
    "elevation_m",
]

COLS_FISICOQUIMICAS = [
    "temperature_deg_c",
    "ph",
    "salinity_psu",
    "oxygen_mg_per_l",
]

COLS_KG = COLS_IDENTIDAD + COLS_EMPO + COLS_ENVO + COLS_GEO + COLS_FISICOQUIMICAS

print(f"Columnas seleccionadas para el KG: {len(COLS_KG)}")
print()
mapping[COLS_KG].head(5)

Columnas seleccionadas para el KG: 23



,#SampleID,study_id,empo_0,empo_1,empo_2,empo_3,env_biome,env_feature,env_material,envo_biome_0,envo_biome_1,envo_biome_2,envo_biome_3,country,latitude_deg,longitude_deg,depth_m,altitude_m,elevation_m,temperature_deg_c,ph,salinity_psu,oxygen_mg_per_l
0,550.L1S116.s.1.sequence,550,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,
1,550.L1S119.s.1.sequence,550,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,
2,550.L1S164.s.1.sequence,550,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,
3,550.L1S194.s.1.sequence,550,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,
4,550.L1S20.s.1.sequence,550,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,


### Distribución EMPO

EMPO (EMP Ontology) es la clasificación jerárquica de entornos propia del EMP.
Tiene cuatro niveles de granularidad creciente: `empo_0`, `empo_1`, `empo_2`,
`empo_3`.

- `empo_0`: distinción más gruesa. Separa muestras de entornos no asociados aorganismos huésped (*free-living*) de las que sí lo están (*host-associated*).
- `empo_3`: clasificación más fina. Por ejemplo: `Soil (non-saline)`, `Seawater (saline)`, `Animal distal gut`.

Para el Knowledge Graph, `empo_3` será el campo principal para definir el nodo `EnvironmentalFeature` de cada muestra.

In [326]:
for nivel in COLS_EMPO:
    counts = mapping[nivel].value_counts()
    print(f"\n{nivel}  ({counts.shape[0]} categorías distintas):")
    for categoria, n in counts.items():
        pct = n / len(mapping) * 100
        print(f"  {n:>5}  ({pct:4.1f}%)  {categoria}")


empo_0  (1 categorías distintas):
   2000  (100.0%)  EMP sample

empo_1  (2 categorías distintas):
   1019  (50.9%)  Host-associated
    981  (49.0%)  Free-living

empo_2  (4 categorías distintas):
    640  (32.0%)  Animal
    595  (29.8%)  Non-saline
    386  (19.3%)  Saline
    379  (18.9%)  Plant

empo_3  (17 categorías distintas):
    143  ( 7.1%)  Animal surface
    129  ( 6.5%)  Soil (non-saline)
    129  ( 6.5%)  Water (non-saline)
    128  ( 6.4%)  Animal distal gut
    128  ( 6.4%)  Animal corpus
    128  ( 6.4%)  Water (saline)
    128  ( 6.4%)  Sediment (saline)
    128  ( 6.4%)  Sediment (non-saline)
    128  ( 6.4%)  Surface (non-saline)
    128  ( 6.4%)  Plant surface
    128  ( 6.4%)  Plant rhizosphere
    128  ( 6.4%)  Animal proximal gut
    123  ( 6.2%)  Plant corpus
    117  ( 5.9%)  Surface (saline)
    113  ( 5.7%)  Animal secretion
     81  ( 4.0%)  Aerosol (non-saline)
     13  ( 0.7%)  Hypersaline (saline)


### Cobertura de campos clave

No todos los campos están rellenos para todas las muestras. Antes de
cruzar con el BIOM conviene saber exactamente qué fracción de muestras
tiene datos en los campos que más nos importan para el KG.

In [327]:
# Valores que consideramos ausentes en el mapping file
MISSING_MARKERS = {
    "",
    "na",
    "n/a",
    "nan",
    "none",
    "null",
    "unknown",
    "missing",
    "not applicable",
    "not provided",
    "not collected",
    "not available",
    "unspecified",
    "no data",
}


def is_missing_value(value) -> bool:
    """
    Devuelve True si un valor representa ausencia de dato.
    """
    if pd.isna(value):
        return True

    clean = str(value).strip()
    clean_lower = clean.lower()

    if clean_lower in MISSING_MARKERS:
        return True

    if clean_lower.startswith("missing:"):
        return True

    return False

print(f"Cobertura de campos clave ({len(mapping):,} muestras totales):\n")
print(f"  {'Campo':<25}  {'Rellenos':>9}  {'Vacíos':>9}  {'Cobertura':>10}")
print(f"  {'-' * 58}")

for col in COLS_KG[2:]:  # saltamos #SampleID y study_id que siempre están
    if col not in mapping.columns:
        print(f"  {col:<25}  {'(no existe)':>30}")
        continue

    n_rellenos = mapping[col].apply(lambda v: not is_missing_value(v)).sum()
    n_vacios   = len(mapping) - n_rellenos
    cobertura  = n_rellenos / len(mapping) * 100

    print(f"  {col:<25}  {n_rellenos:>9,}  {n_vacios:>9,}  {cobertura:>9.1f}%")

Cobertura de campos clave (2,000 muestras totales):

  Campo                       Rellenos     Vacíos   Cobertura
  ----------------------------------------------------------
  empo_0                         2,000          0      100.0%
  empo_1                         2,000          0      100.0%
  empo_2                         2,000          0      100.0%
  empo_3                         2,000          0      100.0%
  env_biome                      2,000          0      100.0%
  env_feature                    2,000          0      100.0%
  env_material                   2,000          0      100.0%
  envo_biome_0                   2,000          0      100.0%
  envo_biome_1                   2,000          0      100.0%
  envo_biome_2                   1,967         33       98.4%
  envo_biome_3                   1,443        557       72.2%
  country                        2,000          0      100.0%
  latitude_deg                   1,998          2       99.9%
  longitude_deg   

## 13. Validar IDs entre BIOM y mapping file

Antes de cruzar los dos archivos, necesitamos confirmar que los IDs de muestra
coinciden exactamente. Es un paso de validación crítico: si hay discrepancias,
el join de la sección 14 produciría filas vacías o perdería muestras sin
ningún aviso visible.

Las causas más comunes de discrepancias en proyectos reales son:

- Diferencias de formato: `"10317.000003428"` vs `10317.000003428` (con o sin comillas).
- Espacios invisibles al inicio o al final del ID.
- Versiones distintas del archivo (un mapping file de una versión distinta al BIOM).
- Muestras filtradas por control de calidad que están en uno pero no en el otro.

En nuestro caso ya sabemos por `inspect_data.py` que los IDs coinciden, pero
la validación explícita aquí cierra el círculo y documenta la comprobación
dentro del notebook.

In [328]:
mapping_ids = set(mapping["#SampleID"])
biom_ids    = set(table.ids(axis="sample"))

solo_mapping = mapping_ids - biom_ids
solo_biom    = biom_ids - mapping_ids
comunes      = mapping_ids & biom_ids

print("Validación de IDs de muestra:")
print(f"  IDs en mapping file : {len(mapping_ids):,}")
print(f"  IDs en BIOM         : {len(biom_ids):,}")
print(f"  IDs comunes         : {len(comunes):,}")
print(f"  Solo en mapping     : {len(solo_mapping):,}")
print(f"  Solo en BIOM        : {len(solo_biom):,}")

Validación de IDs de muestra:
  IDs en mapping file : 2,000
  IDs en BIOM         : 2,000
  IDs comunes         : 2,000
  Solo en mapping     : 0
  Solo en BIOM        : 0


### Verificación formal con assert

Convertimos la validación en una comprobación explícita que detiene
el notebook si algo no cuadra. Es mejor fallar ruidosamente que
continuar con un join silenciosamente incorrecto.

In [329]:
assert len(solo_mapping) == 0, (
    f"Hay {len(solo_mapping)} IDs en el mapping que no están en el BIOM:\n"
    f"  Ejemplos: {list(solo_mapping)[:5]}"
)

assert len(solo_biom) == 0, (
    f"Hay {len(solo_biom)} IDs en el BIOM que no están en el mapping:\n"
    f"  Ejemplos: {list(solo_biom)[:5]}"
)

assert len(comunes) == len(mapping_ids) == len(biom_ids), (
    f"El número de IDs comunes ({len(comunes):,}) no coincide con el total "
    f"de mapping ({len(mapping_ids):,}) o BIOM ({len(biom_ids):,})."
)

print(f"OK: los {len(comunes):,} IDs de muestra coinciden exactamente entre mapping y BIOM.")

OK: los 2,000 IDs de muestra coinciden exactamente entre mapping y BIOM.


### Comprobación adicional: espacios invisibles

Un error silencioso frecuente son los espacios en blanco al inicio o al
final de los IDs. El set de Python los trataría como distintos aunque
visualmente parezcan iguales. Vale la pena comprobarlo explícitamente.

In [330]:
# Comprobar si algún ID del mapping tiene espacios al inicio o al final
ids_con_espacios = [
    sid for sid in mapping["#SampleID"]
    if sid != sid.strip()
]

if ids_con_espacios:
    print(f"AVISO: {len(ids_con_espacios)} IDs tienen espacios invisibles:")
    for sid in ids_con_espacios[:5]:
        print(f"  '{sid}'")
else:
    print("OK: ningún ID del mapping file tiene espacios invisibles.")

# Lo mismo para el BIOM
ids_biom_con_espacios = [
    sid for sid in table.ids(axis="sample")
    if sid != sid.strip()
]

if ids_biom_con_espacios:
    print(f"AVISO: {len(ids_biom_con_espacios)} IDs del BIOM tienen espacios invisibles.")
else:
    print("OK: ningún ID del BIOM tiene espacios invisibles.")

OK: ningún ID del mapping file tiene espacios invisibles.
OK: ningún ID del BIOM tiene espacios invisibles.


## 14. Cruzar mapping file y sample_stats

Tenemos dos tablas con información complementaria sobre las mismas 2.000 muestras:

| Tabla          | Origen       | Qué contiene                                    |
|----------------|--------------|-------------------------------------------------|
| `mapping`      | TSV          | 76 columnas de metadatos ambientales            |
| `sample_stats` | BIOM         | 2 columnas: lecturas totales y ASVs observados  |

Para combinarlas usamos un **join por índice**: renombramos `#SampleID` a
`sample_id`, lo establecemos como índice en el mapping, y hacemos un
`join` con `sample_stats` (que ya tiene `sample_id` como índice).

Usamos `how="inner"` porque ya validamos en la sección 13 que los IDs
coinciden perfectamente. Un inner join sobre dos conjuntos idénticos
no pierde ninguna fila, pero es la opción más segura: si hubiera alguna
discrepancia no detectada, el inner join la haría visible por el número
de filas resultante.

El resultado es el prototipo de `sample_table.csv`: la tabla base de
muestras que usaremos para construir los nodos `Sample` y `Location`
del Knowledge Graph.

In [331]:
# Renombrar #SampleID e indexar
mapping_indexed = (
    mapping
    .rename(columns={"#SampleID": "sample_id"})
    .set_index("sample_id")
)

# Join con sample_stats
sample_table = mapping_indexed.join(sample_stats, how="inner")

print(f"mapping_indexed : {mapping_indexed.shape}")
print(f"sample_stats    : {sample_stats.shape}")
print(f"sample_table    : {sample_table.shape}")
print()

# Verificar que no se ha perdido ninguna muestra
expected_rows = len(comunes)

assert sample_table.shape[0] == expected_rows, (
    f"El join perdió muestras: esperábamos {expected_rows:,}, "
    f"obtenemos {sample_table.shape[0]:,}"
)

print("OK: el join es correcto. No se perdió ninguna muestra.")

sample_table.head()

mapping_indexed : (2000, 75)
sample_stats    : (2000, 2)
sample_table    : (2000, 77)

OK: el join es correcto. No se perdió ninguna muestra.


,BarcodeSequence,LinkerPrimerSequence,Description,host_subject_id,study_id,title,principal_investigator,doi,ebi_accession,target_gene,target_subfragment,pcr_primers,illumina_technology,extraction_center,run_center,run_date,read_length_bp,sequences_split_libraries,observations_closed_ref_greengenes,observations_closed_ref_silva,observations_open_ref_greengenes,observations_deblur_90bp,observations_deblur_100bp,observations_deblur_150bp,emp_release1,qc_filtered,subset_10k,subset_5k,subset_2k,sample_taxid,sample_scientific_name,host_taxid,host_common_name_provided,host_common_name,host_scientific_name,host_superkingdom,host_kingdom,host_phylum,host_class,host_order,host_family,host_genus,host_species,collection_timestamp,country,latitude_deg,longitude_deg,depth_m,altitude_m,elevation_m,env_biome,env_feature,env_material,envo_biome_0,envo_biome_1,envo_biome_2,envo_biome_3,envo_biome_4,envo_biome_5,empo_0,empo_1,empo_2,empo_3,adiv_observed_otus,adiv_chao1,adiv_shannon,adiv_faith_pd,temperature_deg_c,ph,salinity_psu,oxygen_mg_per_l,phosphate_umol_per_l,ammonium_umol_per_l,nitrate_umol_per_l,sulfate_umol_per_l,biom_total_reads,biom_observed_asvs
sample_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
550.L1S116.s.1.sequence,ATGCCTGAGCAG,GTGCCAGCMGCCGCGGTAA,sample_20 stool,F4,550,Moving pictures of the human microbiome,Rob Knight,10.1186/gb-2011-12-5-r50,ERP021896,16S rRNA,V4,FWD:GTGCCAGCMGCCGCGGTAA; REV:GGACTACHVGGGTWTCTAAT,HiSeq,CCME-Boulder,CGS-GL,8/27/10,132,33383,32153,32453,33337,22567,22160,1043,True,True,True,True,True,408170,human gut metagenome,9606,human,human,Homo sapiens,sk__Eukaryota,k__Metazoa,p__Chordata,c__Mammalia,o__Primates,f__Hominidae,g__Homo,s__Homo_sapiens,2009-03-29 00:00:00,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,urban biome,,EMP sample,Host-associated,Animal,Animal distal gut,93,114.375,3.8674139942279702,12.4579888558,,,,,,,,,5000,91
550.L1S119.s.1.sequence,CAGCACTAAGCG,GTGCCAGCMGCCGCGGTAA,sample_23 stool,F4,550,Moving pictures of the human microbiome,Rob Knight,10.1186/gb-2011-12-5-r50,ERP021896,16S rRNA,V4,FWD:GTGCCAGCMGCCGCGGTAA; REV:GGACTACHVGGGTWTCTAAT,HiSeq,CCME-Boulder,CGS-GL,8/27/10,132,40944,39472,39929,40870,27871,27191,1272,True,True,True,True,True,408170,human gut metagenome,9606,human,human,Homo sapiens,sk__Eukaryota,k__Metazoa,p__Chordata,c__Mammalia,o__Primates,f__Hominidae,g__Homo,s__Homo_sapiens,2009-04-06 00:00:00,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,urban biome,,EMP sample,Host-associated,Animal,Animal distal gut,82,116.5,3.2651640717372468,10.719448075849998,,,,,,,,,5000,77
550.L1S164.s.1.sequence,ATGTACGGCGAC,GTGCCAGCMGCCGCGGTAA,sample_73 stool,M3,550,Moving pictures of the human microbiome,Rob Knight,10.1186/gb-2011-12-5-r50,ERP021896,16S rRNA,V4,FWD:GTGCCAGCMGCCGCGGTAA; REV:GGACTACHVGGGTWTCTAAT,HiSeq,CCME-Boulder,CGS-GL,8/27/10,132,35636,34550,34666,35599,24134,23686,1161,True,True,True,True,True,408170,human gut metagenome,9606,human,human,Homo sapiens,sk__Eukaryota,k__Metazoa,p__Chordata,c__Mammalia,o__Primates,f__Hominidae,g__Homo,s__Homo_sapiens,2008-11-21 00:00:00,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,urban biome,,EMP sample,Host-associated,Animal,Animal distal gut,108,135.06666666666666,3.6611236852550224,14.214157823302994,,,,,,,,,5000,104
550.L1S194.s.1.sequence,CGAAGACTGCTG,GTGCCAGCMGCCGCGGTAA,sample_105 stool,M3,550,Moving pictures of the human microbiome,Rob Knight,10.1186/gb-2011-12-5-r50,ERP021896,16S rRNA,V4,FWD:GTGCCAGCMGCCGCGGTAA; REV:GGACTACHVGGGTWTCTAAT,HiSeq,CCME-Boulder,CGS-GL,8/27/10,132,46992,43925,43852,46875,30041

### Seleccionar y ordenar columnas relevantes para el KG

La tabla completa tiene 77 columnas, muchas de las cuales no son relevantes para el Knowledge Graph (metadatos de secuenciación, campos administrativos, métricas de control de calidad...).

Seleccionamos las columnas que corresponden a los nodos y relaciones que construiremos en GraphDB, añadiendo las dos nuevas columnas del BIOM al final.

In [332]:
COLS_PREVIEW = (
    COLS_EMPO
    + COLS_ENVO
    + COLS_GEO
    + COLS_FISICOQUIMICAS
    + ["biom_total_reads", "biom_observed_asvs"]
)

sample_table_preview = sample_table[COLS_PREVIEW].copy()

print(f"Columnas seleccionadas para el KG : {len(COLS_PREVIEW)}")
print(f"Shape de sample_table_preview          : {sample_table_preview.shape}")
print()
sample_table_preview.head()

Columnas seleccionadas para el KG : 23
Shape de sample_table_preview          : (2000, 23)



,empo_0,empo_1,empo_2,empo_3,env_biome,env_feature,env_material,envo_biome_0,envo_biome_1,envo_biome_2,envo_biome_3,country,latitude_deg,longitude_deg,depth_m,altitude_m,elevation_m,temperature_deg_c,ph,salinity_psu,oxygen_mg_per_l,biom_total_reads,biom_observed_asvs
sample_id,,,,,,,,,,,,,,,,,,,,,,,
550.L1S116.s.1.sequence,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,,5000,91
550.L1S119.s.1.sequence,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,,5000,77
550.L1S164.s.1.sequence,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,,5000,104
550.L1S194.s.1.sequence,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,,5000,115
550.L1S20.s.1.sequence,EMP sample,Host-associated,Animal,Animal distal gut,urban biome,human-associated habitat,feces,biome,terrestrial biome,anthropogenic terrestrial biome,dense settlement biome,GAZ:United States of America,40.015,-105.271,0,0.0,1624.1,,,,,5000,80


### Cobertura combinada de la tabla final

Ahora que tenemos todas las columnas juntas, podemos ver cuántas muestras
tienen datos completos en los campos más importantes para el KG.

Esto nos ayuda a dimensionar cuántos nodos de cada tipo podremos crear:
- Un nodo `Location` requiere `latitude_deg` y `longitude_deg`.
- Un nodo `EnvironmentalFeature` requiere al menos `empo_3` o `envo_biome_0`.
- Una relación `has_feature` con pH requiere el campo `ph`.

In [ ]:
CAMPOS_CRITICOS = {
    "Nodo Location"          : ["latitude_deg", "longitude_deg"],
    "Nodo EnvFeature (EMPO)" : ["empo_3"],
    "Nodo EnvFeature (ENVO)" : ["envo_biome_0"],
    "Propiedad pH"           : ["ph"],
    "Propiedad temperatura"  : ["temperature_deg_c"],
    "Propiedad salinidad"    : ["salinity_psu"],
}

print(f"Cobertura de campos críticos para el KG ({len(sample_table_preview):,} muestras):\n")
print(f"  {'Componente KG':<28}  {'Campos requeridos':<35}  {'Muestras':>9}  {'Cobertura':>10}")
print(f"  {'-' * 88}")

for componente, campos in CAMPOS_CRITICOS.items():
    # Una muestra cubre este componente si TODOS sus campos requeridos tienen un valor no ausente
    mascara = pd.Series([True] * len(sample_table_preview), index=sample_table_preview.index)

    for campo in campos:
        if campo in sample_table_preview.columns:
            mascara &= ~sample_table_preview[campo].apply(is_missing_value)

    n_cubre   = mascara.sum()
    cobertura = n_cubre / len(sample_table_preview) * 100
    campos_str = ", ".join(campos)

    print(f"  {componente:<28}  {campos_str:<35}  {n_cubre:>9,}  {cobertura:>9.1f}%")

Cobertura de campos críticos para el KG (2,000 muestras):

  Componente KG                 Campos requeridos                     Muestras   Cobertura
  ----------------------------------------------------------------------------------------
  Nodo Location                 latitude_deg, longitude_deg              1,998       99.9%
  Nodo EnvFeature (EMPO)        empo_3                                   2,000      100.0%
  Nodo EnvFeature (ENVO)        envo_biome_0                             2,000      100.0%
  Propiedad pH                  ph                                         284       14.2%
  Propiedad temperatura         temperature_deg_c                          411       20.5%
  Propiedad salinidad           salinity_psu                               121        6.0%


### Vista previa de las muestras con datos más completos

Para el KG nos interesan especialmente las muestras con más campos rellenos, ya que generarán nodos y relaciones más ricos. Identificamos cuáles son.

Las muestras con más campos rellenos suelen ser de estudios bien documentados de suelos o agua de mar, donde se midieron parámetros fisicoquímicos además de la taxonomía. Estas muestras serán las más informativas para el KG.

In [334]:
# Contar cuántos campos clave tiene rellenos cada muestra
sample_table_preview["n_campos_rellenos"] = sample_table_preview[COLS_PREVIEW[:-2]].apply(
    lambda row: sum(not is_missing_value(v) for v in row),
    axis=1,
)

print("Distribución de campos rellenos por muestra:")
print(sample_table_preview["n_campos_rellenos"].describe().round(1))
print()
print("Las 5 muestras con más campos rellenos:")
print(
    sample_table_preview[["empo_3", "country", "ph", "temperature_deg_c", "n_campos_rellenos"]]
    .nlargest(5, "n_campos_rellenos")
    .to_string()
)

# Limpiar columna auxiliar
sample_table_preview = sample_table_preview.drop(columns=["n_campos_rellenos"])

Distribución de campos rellenos por muestra:
count    2000.0
mean       16.9
std         1.0
min        14.0
25%        16.0
50%        17.0
75%        17.0
max        20.0
Name: n_campos_rellenos, dtype: float64

Las 5 muestras con más campos rellenos:
                                           empo_3      country    ph temperature_deg_c  n_campos_rellenos
sample_id                                                                                                
945.P6.B1.lane2.NoIndex.L002   Water (non-saline)  GAZ:Germany  6.71              17.4                 20
945.P6.H7.lane2.NoIndex.L002   Water (non-saline)  GAZ:Germany  4.86               4.3                 20
945.P13.B3.lane5.NoIndex.L005  Water (non-saline)  GAZ:Germany  8.55               4.8                 20
945.P9.C9.lane3.NoIndex.L003   Water (non-saline)  GAZ:Germany  8.75              19.5                 20
945.P14.F9.lane3.NoIndex.L003  Water (non-saline)  GAZ:Germany  8.35              11.7                 20


## 15. Exploración ambiental básica

Con `sample_table_preview` ya podemos hacer las primeras preguntas ecológicas
sobre el dataset. No es análisis estadístico riguroso — es exploración para
entender la estructura de los datos antes de construir el KG.

Dos preguntas concretas:

1. ¿Cómo varía la diversidad microbiana entre tipos de entorno (`empo_3`)?
2. ¿Qué ASVs dominan en una muestra concreta, y a qué taxones pertenecen?

Estas mismas preguntas, una vez el KG esté construido, se responderán
con consultas SPARQL en GraphDB en lugar de con pandas.

In [337]:
empo_diversity = (
    sample_table
    .groupby("empo_3")["biom_observed_asvs"]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_values("median", ascending=False)
)

empo_diversity.round(1)

,count,mean,median,min,max
empo_3,,,,,
Plant rhizosphere,128,1299.9,1416.5,207,1992
Sediment (non-saline),128,1053.8,1163.5,80,1824
Soil (non-saline),129,841.7,855.0,44,2007
Sediment (saline),128,654.9,636.5,13,1657
Surface (non-saline),128,434.6,403.5,17,1365
Water (non-saline),129,505.4,368.0,9,2438
Water (saline),128,279.2,231.5,73,992
Aerosol (non-saline),81,283.0,220.0,27,972
Surface (saline),117,348.6,216.0,37,1896


Esta tabla permite comparar la riqueza observada de ASVs entre categorías ambientales EMPO. Todavía no es un análisis estadístico formal, pero sirve para comprobar que la unión BIOM + mapping file ya permite formular preguntas ecológicas.

In [338]:
sample_id = sample_stats["biom_observed_asvs"].idxmax()

values = table.data(sample_id, axis="sample", dense=True)

sample_abundance = pd.DataFrame({
    "asv_id": observation_ids,
    "count": values,
})

sample_abundance = (
    sample_abundance
    .query("count > 0")
    .sort_values("count", ascending=False)
)

print(f"Muestra seleccionada: {sample_id}")
print(f"ASVs presentes: {sample_abundance.shape[0]:,}")
print(f"Total lecturas: {sample_abundance['count'].sum():.0f}")

sample_abundance.head(10)

Muestra seleccionada: 1883.2010.21.Crump.Artic.LTREB.main.lane3.NoIndex
ASVs presentes: 2,438
Total lecturas: 5000


,asv_id,count
123396,TACGTAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTATATAAGACAGATGTGAAATCCCCGGGCTCAAC,149.0
24506,TACGTAGGGTGCGAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTATATAAGACAGATGTGAAATCCCCGGGCTCAAC,110.0
133921,TACGTAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTCCGCTAAGACAGATGTGAAATCCCCGGGCTTAAC,90.0
130817,TACAGAGGTGGCAAGCGTTGTTCGGATTTATTGGGCGTAAAGGGTCCGTAGGCGGCCTGAAAAGTTGGATGTGAAATCCCACAGCTTAAC,43.0
140379,TACGTAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTATGCAAGACAGATGTGAAATCCCCGGGCTCAAC,43.0
119280,TACGAAGGGGGCTAGCGTTGCTCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGCCGATTAAGTCGGGGGTGAAAGCCTGTGGCTCAAC,41.0
6860,TACGTAGGGTCCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTGTGCAAGACCGATGTGAAATCCCCGAGCTTAAC,36.0
99037,TACGAAGGGGGCTAGCGTTGCTCGGAATCACTGGGCGTAAAGGGTGCGTAGGCGGGTCTTTAAGTCAGGGGTGAAATCCTGGAGCTCAAC,32.0
29149,TACAGAGGTGGCAAGCGTTGTTCGGATTTATTGGGCGTAAAGGGTCCGTAGGCGGCCTGGAAAGTTGGATGTGAAATCCCACAGCTTAAC,29.0
47202,TACGGAGGGAGCTAGCGTTATTCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGCTTTGTAAGTTAGAGGTGAAAGCCTGGAGCTCAAC,25.0


In [339]:
top_asvs = sample_abundance.head(10).copy()

tax_rows = []

for asv_id in top_asvs["asv_id"]:
    meta = table.metadata(id=asv_id, axis="observation")
    tax_raw = meta.get("taxonomy", [])
    tax_clean = [t[3:] if len(t) > 3 else "" for t in tax_raw]
    tax_clean += [""] * (len(TAX_LEVELS) - len(tax_clean))
    tax_rows.append(dict(zip(TAX_LEVELS, tax_clean)))

top_asvs_with_taxonomy = pd.concat(
    [
        top_asvs.reset_index(drop=True),
        pd.DataFrame(tax_rows),
    ],
    axis=1,
)

top_asvs_with_taxonomy

,asv_id,count,kingdom,phylum,class,order,family,genus,species
0,TACGTAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTATATAAGACAGATGTGAAATCCCCGGGCTCAAC,149.0,Bacteria,Proteobacteria,Betaproteobacteria,Burkholderiales,Comamonadaceae,,
1,TACGTAGGGTGCGAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTATATAAGACAGATGTGAAATCCCCGGGCTCAAC,110.0,Bacteria,Proteobacteria,Betaproteobacteria,Burkholderiales,Comamonadaceae,,
2,TACGTAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTCCGCTAAGACAGATGTGAAATCCCCGGGCTTAAC,90.0,Bacteria,Proteobacteria,Betaproteobacteria,Burkholderiales,Burkholderiaceae,Burkholderia,bryophila
3,TACAGAGGTGGCAAGCGTTGTTCGGATTTATTGGGCGTAAAGGGTCCGTAGGCGGCCTGAAAAGTTGGATGTGAAATCCCACAGCTTAAC,43.0,Bacteria,OP3,PBS-25,,,,
4,TACGTAGGGTGCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTATGCAAGACAGATGTGAAATCCCCGGGCTCAAC,43.0,Bacteria,Proteobacteria,Betaproteobacteria,Burkholderiales,Comamonadaceae,Paucibacter,
5,TACGAAGGGGGCTAGCGTTGCTCGGAATCACTGGGCGTAAAGGGCGCGTAGGCGGCCGATTAAGTCGGGGGTGAAAGCCTGTGGCTCAAC,41.0,Bacteria,Proteobacteria,Alphaproteobacteria,Rhizobiales,Methylobacteriaceae,,
6,TACGTAGGGTCCAAGCGTTAATCGGAATTACTGGGCGTAAAGCGTGCGCAGGCGGTTGTGCAAGACCGATGTGAAATCCCCGAGCTTAAC,36.0,Bacteria,Proteobacteria,Betaproteobacteria,Burkholderiales,Oxalobacteraceae,Ralstonia,
7,TACGAAGGGGGCTAGCGTTGCTCGGAATCACTGGGCGTAAAGGGTGCGTAGGCGGGTCTTTAAGTCAGGGGTGAAATCCTGGAGCTCAAC,32.0,Bacteria,Proteobacteria,Alphaproteobacteria,Rhizobiales,Bradyrhizobiaceae,Bradyrhizobium,
8,TACAGAGGTGGCAAGCGTTGTTCGGATTTATTGGGCGTAAAGGGTCCGTAGGCGGCCTGGAAAGTTGGATGTGAAATCCCACAGCTTAAC,29.0,Bacteria,OP3,PBS-25,,,,
9,TACGGAGGGAGCTAGCGTTATTCGGAATTACTGGGCGTAAAGCGCACGTAGGCGGCTTTGTAAGTTAGAGGTGAAAGCCTGGAGCTCAAC,25.0,Bacteria,Proteobacteria,Alphaproteobacteria,Sphingomonadales,Sphingomonadaceae,Sphingomonas,


## Conclusiones del notebook

En este notebook hemos comprobado que:

- El archivo BIOM se carga correctamente con `biom-format`.
- La tabla contiene 155.002 ASVs y 2.000 muestras.
- La matriz es muy dispersa, como es habitual en datos de microbioma.
- Todas las muestras tienen 5.000 lecturas, coherente con el fichero `rare_5000`.
- Los metadatos de muestra no están dentro del BIOM, sino en el mapping file TSV.
- Los metadatos de observación sí contienen taxonomía.
- Los 2.000 IDs de muestra coinciden exactamente entre BIOM y mapping file.
- La tabla `sample_stats` puede cruzarse correctamente con el mapping file.
- El resultado `sample_table` es el prototipo de la futura `sample_table.csv`.

La lógica estable que debería pasar a scripts es:

- Carga del BIOM.
- Validación de IDs.
- Construcción de `sample_stats`.
- Join con mapping file.
- Exportación de `sample_table.csv`.

## Qué lógica pasará a scripts

Este notebook es exploratorio. No todo debe copiarse tal cual a producción.

### Pasará a `build_sample_table.py`

- Carga del mapping file.
- Normalización de valores ausentes.
- Conversión de columnas numéricas.
- Carga del BIOM.
- Validación de IDs.
- Construcción de `sample_stats`.
- Join mapping + BIOM.
- Exportación de `data/processed/sample_table.csv`.

### Podría pasar a `src/empkg/io/biom.py`

- `load_biom_table`
- `build_biom_sample_stats`
- `extract_taxonomy_preview`
- `extract_abundance_for_sample`

### Se queda solo en el notebook

- Impresiones didácticas.
- Exploración HDF5 detallada.
- Tablas descriptivas.
- Explicaciones largas de los ejes y del formato.